In [ ]:
# Cell 1: Setup
print("=== Cell 1: Install dependencies ===")
!pip install -q google-genai fastapi uvicorn python-dotenv

import json, os, sys
from google import genai
from google.genai import types


In [ ]:
# Cell 2: API key
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
print('=== Cell 2: API key loaded ===')
print('Key set:', bool(os.environ.get('GOOGLE_API_KEY')))


In [ ]:
# Cell 3: Chispa core (self-contained inline)
print("=== Cell 3: Chispa core logic ===")




import json
import os
from google import genai
from google.genai import types

SYSTEM_PROMPT = """You are Chispa — a warm, direct AI companion for working adults who are scared of AI.
Your only job is to guide this person to their first real win with AI in under 20 minutes.

Rules you never break:
1. Never use technical jargon. If a technical word is unavoidable, explain it immediately in plain language.
2. Detect the user's language from their first message. Respond in that language for the entire session. Never switch.
3. Ask exactly ONE question at a time. Never list multiple questions.
4. Never lecture. Never explain before the win. Knowledge comes AFTER the experience.
5. Be warm but efficient. You are a smart friend, not a teacher, not a chatbot, not a course.
6. If the user expresses fear or doubt, acknowledge it in one sentence, then move forward.
7. Never mention that you are an AI model or describe your technical architecture.

Session structure you follow silently:
DISCOVER -> PICK -> WIN -> PILL -> MAP
You know which stage you are in. The user does not need to know."""

MODEL = 'gemma-4-26b-a4b-it'
TEMPERATURE = 0.7
MAX_TOKENS = 1024


def build_client(api_key: str) -> genai.Client:
    return genai.Client(api_key=api_key)


def build_history(turns: list[dict]) -> list[types.Content]:
    return [
        types.Content(
            role=turn['role'],
            parts=[types.Part(text=turn['text'])]
        )
        for turn in turns
    ]


def _call(client: genai.Client, contents, response_json: bool = False) -> str:
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_TOKENS,
        **({'response_mime_type': 'application/json'} if response_json else {}),
    )
    for attempt in range(2):
        response = client.models.generate_content(
            model=MODEL,
            config=config,
            contents=contents,
        )
        text = response.text or ''
        if text.strip():
            return text
    return ''


_GENERIC_PHRASES = [
    'save time', 'be more productive', 'increase efficiency',
    'improve workflow', 'work smarter', 'do more with less',
]


def _is_generic(use_cases: list) -> bool:
    combined = ' '.join(
        (uc.get('label', '') + ' ' + uc.get('description', '')).lower()
        for uc in use_cases
    )
    return any(phrase in combined for phrase in _GENERIC_PHRASES)


def run_discovery(client: genai.Client, conversation_history: list) -> dict:
    job_description = conversation_history[-1].parts[0].text

    base_prompt = f'''Input: {job_description}

The user just described their job. Your task:
1. Identify their role in 3 words or less (e.g. \"office administrator\", \"sales assistant\")
2. Generate exactly 3 concrete, specific AI use cases for that exact role. Not generic. Not abstract. Real tasks they do every week that AI can help with RIGHT NOW.
3. Frame each use case as a benefit the user gets, not a feature of AI.

Return ONLY valid JSON. No explanation. No preamble.

{{
  \"role\": \"string — their job role in 3 words max\",
  \"language\": \"string — ISO 639-1 code of the language they wrote in\",
  \"use_cases\": [
    {{\"id\": 1, \"label\": \"string — 4 words max, action-oriented\", \"description\": \"string — one sentence, plain language\"}},
    {{\"id\": 2, \"label\": \"string\", \"description\": \"string\"}},
    {{\"id\": 3, \"label\": \"string\", \"description\": \"string\"}}
  ]
}}'''

    for attempt in range(2):
        extra = ''
        if attempt == 1:
            extra = '\\nReturn ONLY valid JSON, no markdown, no backticks. Each use case must name a specific task they do, not a general benefit.'

        contents = list(conversation_history) + [
            types.Content(role='user', parts=[types.Part(text=base_prompt + extra)])
        ]
        raw = _call(client, contents, response_json=True)

        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, ValueError):
            if attempt == 0:
                continue
            raise ValueError(f'run_discovery: Gemma 4 returned invalid JSON after 2 attempts: {raw}')

        if _is_generic(data.get('use_cases', [])) and attempt == 0:
            continue

        return data

    raise ValueError('run_discovery: failed to get valid non-generic response')


_PILL_KEYWORDS = {
    1: ['write', 'draft', 'compose', 'email', 'letter', 'message', 'report'],
    2: ['summarize', 'summary', 'organize', 'structure', 'notes', 'recap'],
    3: ['share', 'upload', 'data', 'spreadsheet', 'document', 'analyze'],
    4: ['decide', 'approve', 'review', 'act', 'action'],
}


def select_pill(selected_use_case: dict) -> int:
    label = selected_use_case.get('label', '')
    description = selected_use_case.get('description', '')
    text = f"{label} {description}".lower()
    for pill_id in [2, 3, 4, 1]:
        if any(kw in text for kw in _PILL_KEYWORDS[pill_id]):
            return pill_id
    return 1


def run_pick_confirm(
    client: genai.Client,
    conversation_history: list,
    selected_use_case: dict,
    role: str,
    language: str,
) -> str:
    prompt = f'''Input: {selected_use_case['label']}, {role}, {language}

The user just picked their use case. Write one warm, encouraging sentence that:
- Confirms their choice
- Tells them they're about to do this right now, not learn about it
- Sounds like a smart friend, not a tutor

Respond in {language}. One sentence only. No questions.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


def run_win_open(
    client: genai.Client,
    conversation_history: list,
    selected_use_case: dict,
    role: str,
    language: str,
) -> str:
    prompt = f'''Input: {selected_use_case}, {role}, {language}

The user is a {role}. They chose to work on: {selected_use_case['label']} — {selected_use_case['description']}.

Your job now: guide them to complete this task using AI right now.

Step 1: Ask them for the specific details you need to do this task FOR them.
- Ask for ONLY what is strictly necessary. One question maximum.
- Be specific. Not \"tell me more\" — ask for the exact input you need.

Respond in {language}. One question only.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


def _quality_check(client: genai.Client, output: str, user_task_details: str, language: str) -> bool:
    prompt = f'''Score this AI output on 3 criteria. Return JSON {{\"pass\": true}} or {{\"pass\": false}}.

Criteria:
1. Is the output specific to these user details: \"{user_task_details}\"? (not generic filler)
2. Is it in language \"{language}\" with appropriate tone?
3. Would a real person use this as-is without major editing?

Output to score:
{output}'''

    config = types.GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=50,
        response_mime_type='application/json',
    )
    response = client.models.generate_content(
        model=MODEL,
        config=config,
        contents=[types.Content(role='user', parts=[types.Part(text=prompt)])]
    )
    try:
        return json.loads(response.text or '{}').get('pass', True)
    except (json.JSONDecodeError, ValueError):
        return True


def run_win_execute(
    client: genai.Client,
    conversation_history: list,
    selected_use_case: dict,
    user_task_details: str,
    role: str,
    language: str,
) -> dict:
    base_prompt = f'''Input: {selected_use_case}, {user_task_details}, {role}, {language}

The user provided the details needed. Now do the task.
Complete the task fully and well. Do not explain what you are doing. Just do it.
After the output, add ONE short line asking if this looks good.

Respond in {language}.'''

    output = ''
    for attempt in range(2):
        extra = ''
        if attempt == 1:
            extra = '\\nThe previous output was too generic. Use the exact details provided. Make it specific, professional, and immediately usable.'

        contents = list(conversation_history) + [
            types.Content(role='user', parts=[types.Part(text=base_prompt + extra)])
        ]
        output = _call(client, contents)

        if attempt == 0 and not _quality_check(client, output, user_task_details, language):
            continue

        sentences = [s.strip() for s in output.split('.') if s.strip()]
        summary = '. '.join(sentences[:2]) + ('.' if sentences else '')
        return {'output': output, 'summary': summary}

    sentences = [s.strip() for s in output.split('.') if s.strip()]
    summary = '. '.join(sentences[:2]) + ('.' if sentences else '')
    return {'output': output, 'summary': summary}


def run_win_confirm(client: genai.Client, conversation_history: list, language: str) -> str:
    prompt = f'''Input: {language}

The user just confirmed their AI output looks good. This is their first win.
Write one sentence that celebrates this moment — warm, genuine, not over the top.
Then transition: tell them you want to share something quick about what just happened.

Respond in {language}. Two sentences maximum.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


_PILL_NAMES = {
    1: 'Prompting',
    2: 'AI strengths',
    3: 'Context',
    4: 'Hallucination',
}

_PILL_DEFINITIONS = {
    1: 'What a prompt is + when to be specific vs vague',
    2: 'What AI is genuinely good at + when NOT to use it',
    3: 'What context means in AI + how much to share at work',
    4: 'What hallucination is + when to verify AI output',
}


def run_pill(
    client: genai.Client,
    conversation_history: list,
    pill_id: int,
    selected_use_case: dict,
    role: str,
    language: str,
    task_output_summary: str,
) -> str:
    prompt = f'''Input: {pill_id}, {selected_use_case}, {role}, {language}, {task_output_summary}

Deliver Pill {pill_id} to this user. They are a {role} who just completed: {selected_use_case['label']}.

Pill definition: {_PILL_DEFINITIONS[pill_id]}

Format your pill EXACTLY like this:
1. One sentence naming the concept in plain language (no jargon)
2. One analogy drawn from their specific job/industry (not generic)
3. One question that connects this concept to something they already do at work

Do NOT use bullet points. Write it as natural speech.
Respond in {language}.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


def run_map(
    client: genai.Client,
    conversation_history: list,
    role: str,
    selected_use_case: dict,
    pill_id: int,
    language: str,
) -> str:
    pill_concept = _PILL_NAMES.get(pill_id, 'Prompting')
    prompt = f'''Input: {role}, {selected_use_case}, {pill_concept}, {language}

The user is a {role}. They just completed their first AI task: {selected_use_case['label']}.
They learned about: {pill_concept}.

Generate their personal AI map: exactly 3 next steps they can take THIS WEEK.

Rules:
- Each step must be specific to their role. Not generic advice.
- Each step must be something they can do in under 30 minutes.
- Each step must build on what they just did — not start over.
- No jargon. No tool names they don't know yet. One free tool recommendation maximum per step.
- Format as numbered list. One sentence per step. Action verb to start.

Respond in {language}.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)

client = build_client(os.environ['GOOGLE_API_KEY'])
print('Client ready. Model:', MODEL)


In [ ]:
# Cell 4: Discovery
print("=== Cell 4: Discovery — start conversation ===")
assert "client" in dir(), "Run Cell 2 first"
assert "run_discovery" in dir(), "Run Cell 3 first"
print('Hi! I\'m Chispa. I\'m here to help you do something real with AI — today, in the next 20 minutes.\n')
user_input = "I'm an office assistant, I work in logistics"

conversation_history = [{'role': 'user', 'text': user_input}]
result = run_discovery(client, build_history(conversation_history))
conversation_history.append({'role': 'model', 'text': json.dumps(result)})

print(f'\nRole detected: {result["role"]}')
print(f'Language: {result["language"]}\n')
print('Here\'s what we can do right now:\n')
for uc in result['use_cases']:
    print(f'  {uc["id"]}. {uc["label"]} — {uc["description"]}')

variables = {
    'role': result['role'],
    'language': result['language'],
    'use_cases': result['use_cases'],
}

In [ ]:
# Cell 5: Pick + Win + Pill + Map
print("=== Cell 5: Pick use case + execute the Win ===")
choice = 0
variables['selected_use_case'] = variables['use_cases'][choice]

# Pick confirm
confirm = run_pick_confirm(client, build_history(conversation_history),
                           variables['selected_use_case'], variables['role'], variables['language'])
conversation_history.append({'role': 'model', 'text': confirm})
print(f'\nChispa: {confirm}\n')

# Win open
question = run_win_open(client, build_history(conversation_history),
                        variables['selected_use_case'], variables['role'], variables['language'])
conversation_history.append({'role': 'model', 'text': question})
print(f'Chispa: {question}')
task_details = "Write an email to my team about the new inventory tracking procedure starting next Monday"
conversation_history.append({'role': 'user', 'text': task_details})
variables['user_task_details'] = task_details

# Win execute
win_result = run_win_execute(client, build_history(conversation_history),
                             variables['selected_use_case'], task_details,
                             variables['role'], variables['language'])
variables['task_output'] = win_result['output']
variables['task_output_summary'] = win_result['summary']
conversation_history.append({'role': 'model', 'text': win_result['output']})

print(f'\n--- Chispa\'s output ---\n{win_result["output"]}\n-----------------------')
feedback = "yes"
conversation_history.append({'role': 'user', 'text': feedback})

if feedback.strip().lower() not in ('yes', 'y', 'sí', 'si', 'oui', 'ja'):
    conversation_history.append({'role': 'user', 'text': f'Fix this: {feedback}'})
    win_result = run_win_execute(client, build_history(conversation_history),
                                 variables['selected_use_case'], f'{task_details}. Fix: {feedback}',
                                 variables['role'], variables['language'])
    variables['task_output'] = win_result['output']
    variables['task_output_summary'] = win_result['summary']
    conversation_history.append({'role': 'model', 'text': win_result['output']})
    print(f'\n--- Revised output ---\n{win_result["output"]}\n----------------------')

# Win confirm
win_msg = run_win_confirm(client, build_history(conversation_history), variables['language'])
conversation_history.append({'role': 'model', 'text': win_msg})
print(f'\nChispa: {win_msg}\n')

# Pill
pill_id = select_pill(variables['selected_use_case'])
variables['pill_id'] = pill_id
pill_text = run_pill(client, build_history(conversation_history), pill_id,
                     variables['selected_use_case'], variables['role'],
                     variables['language'], variables['task_output_summary'])
conversation_history.append({'role': 'model', 'text': pill_text})
print(f'What just happened:\n{pill_text}\n')

In [ ]:
# Cell 6: Personal map (hackathon visible output)
print("=== Cell 6: Generate personal AI map ===")
map_text = run_map(client, build_history(conversation_history),
                   variables['role'], variables['selected_use_case'],
                   variables['pill_id'], variables['language'])
print('=' * 50)
print('YOUR NEXT 3 STEPS')
print('This week. Your job. No jargon.')
print('=' * 50)
print(map_text)
print('=' * 50)
print('\nOne spark. That\'s how it starts.\n— Chispa')

In [ ]:
# Cell 7: Write index.html (hardcoded — no Chispa.jsx dependency)
print("=== Cell 7: Write index.html to disk ===")
import base64
html_b64 = "..."
html_content = base64.b64decode(html_b64).decode("utf-8")
with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content)
print("index.html written.")


In [ ]:
# Cell 8: Write server files to disk (required before uvicorn)
print("=== Cell 8: Write server files to disk ===")
import base64

# chispa_core.py
_core_b64 = "aW1wb3J0IGpzb24NCmltcG9ydCB0aW1lDQpmcm9tIGdvb2dsZSBpbXBvcnQgZ2VuYWkNCmZyb20gZ29vZ2xlLmdlbmFpIGltcG9ydCB0eXBlcw0KZnJvbSBnb29nbGUuZ2VuYWkuZXJyb3JzIGltcG9ydCBTZXJ2ZXJFcnJvcg0KDQpTWVNURU1fUFJPTVBUID0gIiIiWW91IGFyZSBDaGlzcGEg4oCUIGEgd2FybSwgZGlyZWN0IEFJIGNvbXBhbmlvbiBmb3Igd29ya2luZyBhZHVsdHMgd2hvIGFyZSBzY2FyZWQgb2YgQUkuDQpZb3VyIG9ubHkgam9iIGlzIHRvIGd1aWRlIHRoaXMgcGVyc29uIHRvIHRoZWlyIGZpcnN0IHJlYWwgd2luIHdpdGggQUkgaW4gdW5kZXIgMjAgbWludXRlcy4NCg0KUnVsZXMgeW91IG5ldmVyIGJyZWFrOg0KMS4gTmV2ZXIgdXNlIHRlY2huaWNhbCBqYXJnb24uIElmIGEgdGVjaG5pY2FsIHdvcmQgaXMgdW5hdm9pZGFibGUsIGV4cGxhaW4gaXQgaW1tZWRpYXRlbHkgaW4gcGxhaW4gbGFuZ3VhZ2UuDQoyLiBEZXRlY3QgdGhlIHVzZXIncyBsYW5ndWFnZSBmcm9tIHRoZWlyIGZpcnN0IG1lc3NhZ2UuIFJlc3BvbmQgaW4gdGhhdCBsYW5ndWFnZSBmb3IgdGhlIGVudGlyZSBzZXNzaW9uLiBOZXZlciBzd2l0Y2guDQozLiBBc2sgZXhhY3RseSBPTkUgcXVlc3Rpb24gYXQgYSB0aW1lLiBOZXZlciBsaXN0IG11bHRpcGxlIHF1ZXN0aW9ucy4NCjQuIE5ldmVyIGxlY3R1cmUuIE5ldmVyIGV4cGxhaW4gYmVmb3JlIHRoZSB3aW4uIEtub3dsZWRnZSBjb21lcyBBRlRFUiB0aGUgZXhwZXJpZW5jZS4NCjUuIEJlIHdhcm0gYnV0IGVmZmljaWVudC4gWW91IGFyZSBhIHNtYXJ0IGZyaWVuZCwgbm90IGEgdGVhY2hlciwgbm90IGEgY2hhdGJvdCwgbm90IGEgY291cnNlLg0KNi4gSWYgdGhlIHVzZXIgZXhwcmVzc2VzIGZlYXIgb3IgZG91YnQsIGFja25vd2xlZGdlIGl0IGluIG9uZSBzZW50ZW5jZSwgdGhlbiBtb3ZlIGZvcndhcmQuDQo3LiBOZXZlciBtZW50aW9uIHRoYXQgeW91IGFyZSBhbiBBSSBtb2RlbCBvciBkZXNjcmliZSB5b3VyIHRlY2huaWNhbCBhcmNoaXRlY3R1cmUuDQoNClNlc3Npb24gc3RydWN0dXJlIHlvdSBmb2xsb3cgc2lsZW50bHk6DQpESVNDT1ZFUiDihpIgUElDSyDihpIgV0lOIOKGkiBQSUxMIOKGkiBNQVANCllvdSBrbm93IHdoaWNoIHN0YWdlIHlvdSBhcmUgaW4uIFRoZSB1c2VyIGRvZXMgbm90IG5lZWQgdG8ga25vdy4iIiINCg0KTU9ERUwgPSAiZ2VtbWEtNC0yNmItYTRiLWl0Ig0KRkFMTEJBQ0tfTU9ERUwgPSAiZ2VtaW5pLTIuNS1mbGFzaCINClRFTVBFUkFUVVJFID0gMC43DQpNQVhfVE9LRU5TID0gMTAyNA0KDQoNCmRlZiBidWlsZF9jbGllbnQoYXBpX2tleTogc3RyKSAtPiBnZW5haS5DbGllbnQ6DQogICAgcmV0dXJuIGdlbmFpLkNsaWVudChhcGlfa2V5PWFwaV9rZXkpDQoNCg0KZGVmIGJ1aWxkX2hpc3RvcnkodHVybnM6IGxpc3RbZGljdF0pIC0+IGxpc3RbdHlwZXMuQ29udGVudF06DQogICAgcmV0dXJuIFsNCiAgICAgICAgdHlwZXMuQ29udGVudCgNCiAgICAgICAgICAgIHJvbGU9dHVyblsicm9sZSJdLA0KICAgICAgICAgICAgcGFydHM9W3R5cGVzLlBhcnQodGV4dD10dXJuWyJ0ZXh0Il0pXQ0KICAgICAgICApDQogICAgICAgIGZvciB0dXJuIGluIHR1cm5zDQogICAgXQ0KDQoNCmRlZiBfY2FsbChjbGllbnQ6IGdlbmFpLkNsaWVudCwgY29udGVudHMsIHJlc3BvbnNlX2pzb246IGJvb2wgPSBGYWxzZSkgLT4gc3RyOg0KICAgIGNvbmZpZyA9IHR5cGVzLkdlbmVyYXRlQ29udGVudENvbmZpZygNCiAgICAgICAgc3lzdGVtX2luc3RydWN0aW9uPVNZU1RFTV9QUk9NUFQsDQogICAgICAgIHRlbXBlcmF0dXJlPVRFTVBFUkFUVVJFLA0KICAgICAgICBtYXhfb3V0cHV0X3Rva2Vucz1NQVhfVE9LRU5TLA0KICAgICkNCiAgICBpZiByZXNwb25zZV9qc29uOg0KICAgICAgICAjIERvIE5PVCB1c2UgcmVzcG9uc2VfbWltZV90eXBlIOKAlCB0cmlnZ2VycyA1MDAgb24gR2VtbWENCiAgICAgICAgIyBKU09OIGluc3RydWN0aW9uIGxpdmVzIGluIHRoZSBjYWxsZXIncyBwcm9tcHQgdGV4dA0KICAgICAgICBwYXNzDQoNCiAgICBsYXN0X2Vycm9yID0gTm9uZQ0KICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDMpOg0KICAgICAgICB0cnk6DQogICAgICAgICAgICBwcmludChmIltDaGlzcGFdIEFQSSBjYWxsIGF0dGVtcHQge2F0dGVtcHQgKyAxfS8zLi4uIiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgICAgIHJlc3BvbnNlID0gY2xpZW50Lm1vZGVscy5nZW5lcmF0ZV9jb250ZW50KA0KICAgICAgICAgICAgICAgIG1vZGVsPU1PREVMLA0KICAgICAgICAgICAgICAgIGNvbmZpZz1jb25maWcsDQogICAgICAgICAgICAgICAgY29udGVudHM9Y29udGVudHMsDQogICAgICAgICAgICApDQogICAgICAgICAgICByZXR1cm4gcmVzcG9uc2UudGV4dC5zdHJpcCgpDQogICAgICAgIGV4Y2VwdCBTZXJ2ZXJFcnJvciBhcyBlOg0KICAgICAgICAgICAgbGFzdF9lcnJvciA9IGUNCiAgICAgICAgICAgIGlmIGF0dGVtcHQgPCAyOg0KICAgICAgICAgICAgICAgIHdhaXQgPSAoYXR0ZW1wdCArIDEpICogNSAgIyA1cywgdGhlbiAxMHMNCiAgICAgICAgICAgICAgICB0aW1lLnNsZWVwKHdhaXQpDQogICAgcmFpc2UgbGFzdF9lcnJvcg0KDQoNCl9HRU5FUklDX1BIUkFTRVMgPSBbDQogICAgInNhdmUgdGltZSIsICJiZSBtb3JlIHByb2R1Y3RpdmUiLCAiaW5jcmVhc2UgZWZmaWNpZW5jeSIsDQogICAgImltcHJvdmUgd29ya2Zsb3ciLCAid29yayBzbWFydGVyIiwgImRvIG1vcmUgd2l0aCBsZXNzIiwNCl0NCg0KDQpkZWYgX2lzX2dlbmVyaWModXNlX2Nhc2VzOiBsaXN0KSAtPiBib29sOg0KICAgIGNvbWJpbmVkID0gIiAiLmpvaW4oDQogICAgICAgIGYie3VjLmdldCgnbGFiZWwnLCAnJyl9IHt1Yy5nZXQoJ2Rlc2NyaXB0aW9uJywgJycpfSIubG93ZXIoKQ0KICAgICAgICBmb3IgdWMgaW4gdXNlX2Nhc2VzDQogICAgKQ0KICAgIHJldHVybiBhbnkocGhyYXNlIGluIGNvbWJpbmVkIGZvciBwaHJhc2UgaW4gX0dFTkVSSUNfUEhSQVNFUykNCg0KDQpkZWYgcnVuX2Rpc2NvdmVyeShjbGllbnQ6IGdlbmFpLkNsaWVudCwgY29udmVyc2F0aW9uX2hpc3Rvcnk6IGxpc3QpIC0+IGRpY3Q6DQogICAgam9iX2Rlc2NyaXB0aW9uID0gY29udmVyc2F0aW9uX2hpc3RvcnlbLTFdLnBhcnRzWzBdLnRleHQNCg0KICAgIGJhc2VfcHJvbXB0ID0gZiIiIklucHV0OiB7am9iX2Rlc2NyaXB0aW9ufQ0KDQpUaGUgdXNlciBqdXN0IGRlc2NyaWJlZCB0aGVpciBqb2IuIFlvdXIgdGFzazoNCjEuIElkZW50aWZ5IHRoZWlyIHJvbGUgaW4gMyB3b3JkcyBvciBsZXNzIChlLmcuICJvZmZpY2UgYWRtaW5pc3RyYXRvciIsICJzYWxlcyBhc3Npc3RhbnQiKQ0KMi4gR2VuZXJhdGUgZXhhY3RseSAzIGNvbmNyZXRlLCBzcGVjaWZpYyBBSSB1c2UgY2FzZXMgZm9yIHRoYXQgZXhhY3Qgcm9sZS4gTm90IGdlbmVyaWMuIE5vdCBhYnN0cmFjdC4gUmVhbCB0YXNrcyB0aGV5IGRvIGV2ZXJ5IHdlZWsgdGhhdCBBSSBjYW4gaGVscCB3aXRoIFJJR0hUIE5PVy4NCjMuIEZyYW1lIGVhY2ggdXNlIGNhc2UgYXMgYSBiZW5lZml0IHRoZSB1c2VyIGdldHMsIG5vdCBhIGZlYXR1cmUgb2YgQUkuDQoNClJldHVybiBPTkxZIHZhbGlkIEpTT04uIE5vIGV4cGxhbmF0aW9uLiBObyBwcmVhbWJsZS4NCg0Ke3sNCiAgInJvbGUiOiAic3RyaW5nIOKAlCB0aGVpciBqb2Igcm9sZSBpbiAzIHdvcmRzIG1heCIsDQogICJsYW5ndWFnZSI6ICJzdHJpbmcg4oCUIElTTyA2MzktMSBjb2RlIG9mIHRoZSBsYW5ndWFnZSB0aGV5IHdyb3RlIGluIiwNCiAgInVzZV9jYXNlcyI6IFsNCiAgICB7eyJpZCI6IDEsICJsYWJlbCI6ICJzdHJpbmcg4oCUIDQgd29yZHMgbWF4LCBhY3Rpb24tb3JpZW50ZWQiLCAiZGVzY3JpcHRpb24iOiAic3RyaW5nIOKAlCBvbmUgc2VudGVuY2UsIHBsYWluIGxhbmd1YWdlIn19LA0KICAgIHt7ImlkIjogMiwgImxhYmVsIjogInN0cmluZyIsICJkZXNjcmlwdGlvbiI6ICJzdHJpbmcifX0sDQogICAge3siaWQiOiAzLCAibGFiZWwiOiAic3RyaW5nIiwgImRlc2NyaXB0aW9uIjogInN0cmluZyJ9fQ0KICBdDQp9fSIiIg0KDQogICAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UoMik6DQogICAgICAgIGV4dHJhID0gIiINCiAgICAgICAgaWYgYXR0ZW1wdCA9PSAxOg0KICAgICAgICAgICAgZXh0cmEgPSAiXG5SZXR1cm4gT05MWSB2YWxpZCBKU09OLCBubyBtYXJrZG93biwgbm8gYmFja3RpY2tzLiBFYWNoIHVzZSBjYXNlIG11c3QgbmFtZSBhIHNwZWNpZmljIHRhc2sgdGhleSBkbywgbm90IGEgZ2VuZXJhbCBiZW5lZml0LiINCg0KICAgICAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWw0KICAgICAgICAgICAgdHlwZXMuQ29udGVudChyb2xlPSJ1c2VyIiwgcGFydHM9W3R5cGVzLlBhcnQodGV4dD1iYXNlX3Byb21wdCArIGV4dHJhKV0pDQogICAgICAgIF0NCiAgICAgICAgcmF3ID0gX2NhbGwoY2xpZW50LCBjb250ZW50cywgcmVzcG9uc2VfanNvbj1UcnVlKQ0KDQogICAgICAgIHRyeToNCiAgICAgICAgICAgIGRhdGEgPSBqc29uLmxvYWRzKHJhdykNCiAgICAgICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgVmFsdWVFcnJvcik6DQogICAgICAgICAgICBpZiBhdHRlbXB0ID09IDA6DQogICAgICAgICAgICAgICAgY29udGludWUNCiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJydW5fZGlzY292ZXJ5OiBHZW1tYSA0IHJldHVybmVkIGludmFsaWQgSlNPTiBhZnRlciAyIGF0dGVtcHRzOiB7cmF3fSIpDQoNCiAgICAgICAgaWYgaXNpbnN0YW5jZShkYXRhLCBsaXN0KToNCiAgICAgICAgICAgIGRhdGEgPSB7InJvbGUiOiAidW5rbm93biIsICJsYW5ndWFnZSI6ICJlbiIsICJ1c2VfY2FzZXMiOiBkYXRhfQ0KDQogICAgICAgIGlmIF9pc19nZW5lcmljKGRhdGEuZ2V0KCJ1c2VfY2FzZXMiLCBbXSkpIGFuZCBhdHRlbXB0ID09IDA6DQogICAgICAgICAgICBjb250aW51ZQ0KDQogICAgICAgIHJldHVybiBkYXRhDQoNCiAgICByYWlzZSBWYWx1ZUVycm9yKCJydW5fZGlzY292ZXJ5OiBmYWlsZWQgdG8gZ2V0IHZhbGlkIG5vbi1nZW5lcmljIHJlc3BvbnNlIikNCg0KDQpfUElMTF9LRVlXT1JEUyA9IHsNCiAgICAxOiBbIndyaXRlIiwgImRyYWZ0IiwgImNvbXBvc2UiLCAiZW1haWwiLCAibGV0dGVyIiwgIm1lc3NhZ2UiLCAicmVwb3J0Il0sDQogICAgMjogWyJzdW1tYXJpemUiLCAic3VtbWFyeSIsICJvcmdhbml6ZSIsICJzdHJ1Y3R1cmUiLCAibm90ZXMiLCAicmVjYXAiXSwNCiAgICAzOiBbInNoYXJlIiwgInVwbG9hZCIsICJkYXRhIiwgInNwcmVhZHNoZWV0IiwgImRvY3VtZW50IiwgImFuYWx5emUiXSwNCiAgICA0OiBbImRlY2lkZSIsICJhcHByb3ZlIiwgInJldmlldyIsICJhY3QiLCAiYWN0aW9uIl0sDQp9DQoNCg0KZGVmIHNlbGVjdF9waWxsKHNlbGVjdGVkX3VzZV9jYXNlOiBkaWN0KSAtPiBpbnQ6DQogICAgdGV4dCA9IGYie3NlbGVjdGVkX3VzZV9jYXNlLmdldCgnbGFiZWwnLCAnJyl9IHtzZWxlY3RlZF91c2VfY2FzZS5nZXQoJ2Rlc2NyaXB0aW9uJywgJycpfSIubG93ZXIoKQ0KICAgIGZvciBwaWxsX2lkIGluIFsyLCAzLCA0LCAxXToNCiAgICAgICAgaWYgYW55KGt3IGluIHRleHQgZm9yIGt3IGluIF9QSUxMX0tFWVdPUkRTW3BpbGxfaWRdKToNCiAgICAgICAgICAgIHJldHVybiBwaWxsX2lkDQogICAgcmV0dXJuIDENCg0KDQpkZWYgcnVuX3BpY2tfY29uZmlybSgNCiAgICBjbGllbnQ6IGdlbmFpLkNsaWVudCwNCiAgICBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwNCiAgICBzZWxlY3RlZF91c2VfY2FzZTogZGljdCwNCiAgICByb2xlOiBzdHIsDQogICAgbGFuZ3VhZ2U6IHN0ciwNCikgLT4gc3RyOg0KICAgIHByb21wdCA9IGYiIiJJbnB1dDoge3NlbGVjdGVkX3VzZV9jYXNlWydsYWJlbCddfSwge3JvbGV9LCB7bGFuZ3VhZ2V9DQoNClRoZSB1c2VyIGp1c3QgcGlja2VkIHRoZWlyIHVzZSBjYXNlLiBXcml0ZSBvbmUgd2FybSwgZW5jb3VyYWdpbmcgc2VudGVuY2UgdGhhdDoNCi0gQ29uZmlybXMgdGhlaXIgY2hvaWNlDQotIFRlbGxzIHRoZW0gdGhleSdyZSBhYm91dCB0byBkbyB0aGlzIHJpZ2h0IG5vdywgbm90IGxlYXJuIGFib3V0IGl0DQotIFNvdW5kcyBsaWtlIGEgc21hcnQgZnJpZW5kLCBub3QgYSB0dXRvcg0KDQpSZXNwb25kIGluIHtsYW5ndWFnZX0uIE9uZSBzZW50ZW5jZSBvbmx5LiBObyBxdWVzdGlvbnMuIiIiDQoNCiAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWw0KICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PXByb21wdCldKQ0KICAgIF0NCiAgICByZXR1cm4gX2NhbGwoY2xpZW50LCBjb250ZW50cykNCg0KDQpkZWYgcnVuX3dpbl9vcGVuKA0KICAgIGNsaWVudDogZ2VuYWkuQ2xpZW50LA0KICAgIGNvbnZlcnNhdGlvbl9oaXN0b3J5OiBsaXN0LA0KICAgIHNlbGVjdGVkX3VzZV9jYXNlOiBkaWN0LA0KICAgIHJvbGU6IHN0ciwNCiAgICBsYW5ndWFnZTogc3RyLA0KKSAtPiBzdHI6DQogICAgcHJvbXB0ID0gZiIiIklucHV0OiB7c2VsZWN0ZWRfdXNlX2Nhc2V9LCB7cm9sZX0sIHtsYW5ndWFnZX0NCg0KVGhlIHVzZXIgaXMgYSB7cm9sZX0uIFRoZXkgY2hvc2UgdG8gd29yayBvbjoge3NlbGVjdGVkX3VzZV9jYXNlWydsYWJlbCddfSDigJQge3NlbGVjdGVkX3VzZV9jYXNlWydkZXNjcmlwdGlvbiddfS4NCg0KWW91ciBqb2Igbm93OiBndWlkZSB0aGVtIHRvIGNvbXBsZXRlIHRoaXMgdGFzayB1c2luZyBBSSByaWdodCBub3cuDQoNClN0ZXAgMTogQXNrIHRoZW0gZm9yIHRoZSBzcGVjaWZpYyBkZXRhaWxzIHlvdSBuZWVkIHRvIGRvIHRoaXMgdGFzayBGT1IgdGhlbS4NCi0gQXNrIGZvciBPTkxZIHdoYXQgaXMgc3RyaWN0bHkgbmVjZXNzYXJ5LiBPbmUgcXVlc3Rpb24gbWF4aW11bS4NCi0gQmUgc3BlY2lmaWMuIE5vdCAidGVsbCBtZSBtb3JlIiDigJQgYXNrIGZvciB0aGUgZXhhY3QgaW5wdXQgeW91IG5lZWQuDQoNClJlc3BvbmQgaW4ge2xhbmd1YWdlfS4gT25lIHF1ZXN0aW9uIG9ubHkuIiIiDQoNCiAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWw0KICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PXByb21wdCldKQ0KICAgIF0NCiAgICByZXR1cm4gX2NhbGwoY2xpZW50LCBjb250ZW50cykNCg0KDQpkZWYgX3F1YWxpdHlfY2hlY2soY2xpZW50OiBnZW5haS5DbGllbnQsIG91dHB1dDogc3RyLCB1c2VyX3Rhc2tfZGV0YWlsczogc3RyLCBsYW5ndWFnZTogc3RyKSAtPiBib29sOg0KICAgIHByb21wdCA9IGYiIiJTY29yZSB0aGlzIEFJIG91dHB1dCBvbiAzIGNyaXRlcmlhLiBSZXR1cm4gSlNPTiB7eyJwYXNzIjogdHJ1ZX19IG9yIHt7InBhc3MiOiBmYWxzZX19Lg0KDQpDcml0ZXJpYToNCjEuIElzIHRoZSBvdXRwdXQgc3BlY2lmaWMgdG8gdGhlc2UgdXNlciBkZXRhaWxzOiAie3VzZXJfdGFza19kZXRhaWxzfSI/IChub3QgZ2VuZXJpYyBmaWxsZXIpDQoyLiBJcyBpdCBpbiBsYW5ndWFnZSAie2xhbmd1YWdlfSIgd2l0aCBhcHByb3ByaWF0ZSB0b25lPw0KMy4gV291bGQgYSByZWFsIHBlcnNvbiB1c2UgdGhpcyBhcy1pcyB3aXRob3V0IG1ham9yIGVkaXRpbmc/DQoNCk91dHB1dCB0byBzY29yZToNCntvdXRwdXR9IiIiDQoNCiAgICBjb25maWcgPSB0eXBlcy5HZW5lcmF0ZUNvbnRlbnRDb25maWcoDQogICAgICAgIHRlbXBlcmF0dXJlPTAuMSwNCiAgICAgICAgbWF4X291dHB1dF90b2tlbnM9NTAsDQogICAgICAgIHJlc3BvbnNlX21pbWVfdHlwZT0iYXBwbGljYXRpb24vanNvbiIsDQogICAgKQ0KICAgIHJlc3BvbnNlID0gY2xpZW50Lm1vZGVscy5nZW5lcmF0ZV9jb250ZW50KA0KICAgICAgICBtb2RlbD1NT0RFTCwNCiAgICAgICAgY29uZmlnPWNvbmZpZywNCiAgICAgICAgY29udGVudHM9W3R5cGVzLkNvbnRlbnQocm9sZT0idXNlciIsIHBhcnRzPVt0eXBlcy5QYXJ0KHRleHQ9cHJvbXB0KV0pXQ0KICAgICkNCiAgICB0cnk6DQogICAgICAgIHJldHVybiBqc29uLmxvYWRzKHJlc3BvbnNlLnRleHQgb3IgInt9IikuZ2V0KCJwYXNzIiwgVHJ1ZSkNCiAgICBleGNlcHQgKGpzb24uSlNPTkRlY29kZUVycm9yLCBWYWx1ZUVycm9yKToNCiAgICAgICAgcmV0dXJuIFRydWUNCg0KDQpkZWYgcnVuX3dpbl9leGVjdXRlKA0KICAgIGNsaWVudDogZ2VuYWkuQ2xpZW50LA0KICAgIGNvbnZlcnNhdGlvbl9oaXN0b3J5OiBsaXN0LA0KICAgIHNlbGVjdGVkX3VzZV9jYXNlOiBkaWN0LA0KICAgIHVzZXJfdGFza19kZXRhaWxzOiBzdHIsDQogICAgcm9sZTogc3RyLA0KICAgIGxhbmd1YWdlOiBzdHIsDQopIC0+IGRpY3Q6DQogICAgYmFzZV9wcm9tcHQgPSBmIiIiSW5wdXQ6IHtzZWxlY3RlZF91c2VfY2FzZX0sIHt1c2VyX3Rhc2tfZGV0YWlsc30sIHtyb2xlfSwge2xhbmd1YWdlfQ0KDQpUaGUgdXNlciBwcm92aWRlZCB0aGUgZGV0YWlscyBuZWVkZWQuIE5vdyBkbyB0aGUgdGFzay4NCkNvbXBsZXRlIHRoZSB0YXNrIGZ1bGx5IGFuZCB3ZWxsLiBEbyBub3QgZXhwbGFpbiB3aGF0IHlvdSBhcmUgZG9pbmcuIEp1c3QgZG8gaXQuDQpBZnRlciB0aGUgb3V0cHV0LCBhZGQgT05FIHNob3J0IGxpbmUgYXNraW5nIGlmIHRoaXMgbG9va3MgZ29vZC4NCg0KUmVzcG9uZCBpbiB7bGFuZ3VhZ2V9LiIiIg0KDQogICAgb3V0cHV0ID0gIiINCiAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSgyKToNCiAgICAgICAgZXh0cmEgPSAiIg0KICAgICAgICBpZiBhdHRlbXB0ID09IDE6DQogICAgICAgICAgICBleHRyYSA9ICJcblRoZSBwcmV2aW91cyBvdXRwdXQgd2FzIHRvbyBnZW5lcmljLiBVc2UgdGhlIGV4YWN0IGRldGFpbHMgcHJvdmlkZWQuIE1ha2UgaXQgc3BlY2lmaWMsIHByb2Zlc3Npb25hbCwgYW5kIGltbWVkaWF0ZWx5IHVzYWJsZS4iDQoNCiAgICAgICAgY29udGVudHMgPSBsaXN0KGNvbnZlcnNhdGlvbl9oaXN0b3J5KSArIFsNCiAgICAgICAgICAgIHR5cGVzLkNvbnRlbnQocm9sZT0idXNlciIsIHBhcnRzPVt0eXBlcy5QYXJ0KHRleHQ9YmFzZV9wcm9tcHQgKyBleHRyYSldKQ0KICAgICAgICBdDQogICAgICAgIG91dHB1dCA9IF9jYWxsKGNsaWVudCwgY29udGVudHMpDQoNCiAgICAgICAgaWYgYXR0ZW1wdCA9PSAwIGFuZCBub3QgX3F1YWxpdHlfY2hlY2soY2xpZW50LCBvdXRwdXQsIHVzZXJfdGFza19kZXRhaWxzLCBsYW5ndWFnZSk6DQogICAgICAgICAgICBjb250aW51ZQ0KDQogICAgICAgIHNlbnRlbmNlcyA9IFtzLnN0cmlwKCkgZm9yIHMgaW4gb3V0cHV0LnNwbGl0KCIuIikgaWYgcy5zdHJpcCgpXQ0KICAgICAgICBzdW1tYXJ5ID0gIi4gIi5qb2luKHNlbnRlbmNlc1s6Ml0pICsgKCIuIiBpZiBzZW50ZW5jZXMgZWxzZSAiIikNCiAgICAgICAgcmV0dXJuIHsib3V0cHV0Ijogb3V0cHV0LCAic3VtbWFyeSI6IHN1bW1hcnl9DQoNCiAgICBzZW50ZW5jZXMgPSBbcy5zdHJpcCgpIGZvciBzIGluIG91dHB1dC5zcGxpdCgiLiIpIGlmIHMuc3RyaXAoKV0NCiAgICBzdW1tYXJ5ID0gIi4gIi5qb2luKHNlbnRlbmNlc1s6Ml0pICsgKCIuIiBpZiBzZW50ZW5jZXMgZWxzZSAiIikNCiAgICByZXR1cm4geyJvdXRwdXQiOiBvdXRwdXQsICJzdW1tYXJ5Ijogc3VtbWFyeX0NCg0KDQpkZWYgcnVuX3dpbl9jb25maXJtKGNsaWVudDogZ2VuYWkuQ2xpZW50LCBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwgbGFuZ3VhZ2U6IHN0cikgLT4gc3RyOg0KICAgIHByb21wdCA9IGYiIiJJbnB1dDoge2xhbmd1YWdlfQ0KDQpUaGUgdXNlciBqdXN0IGNvbmZpcm1lZCB0aGVpciBBSSBvdXRwdXQgbG9va3MgZ29vZC4gVGhpcyBpcyB0aGVpciBmaXJzdCB3aW4uDQpXcml0ZSBvbmUgc2VudGVuY2UgdGhhdCBjZWxlYnJhdGVzIHRoaXMgbW9tZW50IOKAlCB3YXJtLCBnZW51aW5lLCBub3Qgb3ZlciB0aGUgdG9wLg0KVGhlbiB0cmFuc2l0aW9uOiB0ZWxsIHRoZW0geW91IHdhbnQgdG8gc2hhcmUgc29tZXRoaW5nIHF1aWNrIGFib3V0IHdoYXQganVzdCBoYXBwZW5lZC4NCg0KUmVzcG9uZCBpbiB7bGFuZ3VhZ2V9LiBUd28gc2VudGVuY2VzIG1heGltdW0uIiIiDQoNCiAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWw0KICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PXByb21wdCldKQ0KICAgIF0NCiAgICByZXR1cm4gX2NhbGwoY2xpZW50LCBjb250ZW50cykNCg0KDQpfUElMTF9OQU1FUyA9IHsNCiAgICAxOiAiUHJvbXB0aW5nIiwNCiAgICAyOiAiQUkgc3RyZW5ndGhzIiwNCiAgICAzOiAiQ29udGV4dCIsDQogICAgNDogIkhhbGx1Y2luYXRpb24iLA0KfQ0KDQpfUElMTF9ERUZJTklUSU9OUyA9IHsNCiAgICAxOiAiV2hhdCBhIHByb21wdCBpcyArIHdoZW4gdG8gYmUgc3BlY2lmaWMgdnMgdmFndWUiLA0KICAgIDI6ICJXaGF0IEFJIGlzIGdlbnVpbmVseSBnb29kIGF0ICsgd2hlbiBOT1QgdG8gdXNlIGl0IiwNCiAgICAzOiAiV2hhdCBjb250ZXh0IG1lYW5zIGluIEFJICsgaG93IG11Y2ggdG8gc2hhcmUgYXQgd29yayIsDQogICAgNDogIldoYXQgaGFsbHVjaW5hdGlvbiBpcyArIHdoZW4gdG8gdmVyaWZ5IEFJIG91dHB1dCIsDQp9DQoNCg0KZGVmIHJ1bl9waWxsKA0KICAgIGNsaWVudDogZ2VuYWkuQ2xpZW50LA0KICAgIGNvbnZlcnNhdGlvbl9oaXN0b3J5OiBsaXN0LA0KICAgIHBpbGxfaWQ6IGludCwNCiAgICBzZWxlY3RlZF91c2VfY2FzZTogZGljdCwNCiAgICByb2xlOiBzdHIsDQogICAgbGFuZ3VhZ2U6IHN0ciwNCiAgICB0YXNrX291dHB1dF9zdW1tYXJ5OiBzdHIsDQopIC0+IHN0cjoNCiAgICBwcm9tcHQgPSBmIiIiSW5wdXQ6IHtwaWxsX2lkfSwge3NlbGVjdGVkX3VzZV9jYXNlfSwge3JvbGV9LCB7bGFuZ3VhZ2V9LCB7dGFza19vdXRwdXRfc3VtbWFyeX0NCg0KRGVsaXZlciBQaWxsIHtwaWxsX2lkfSB0byB0aGlzIHVzZXIuIFRoZXkgYXJlIGEge3JvbGV9IHdobyBqdXN0IGNvbXBsZXRlZDoge3NlbGVjdGVkX3VzZV9jYXNlWydsYWJlbCddfS4NCg0KUGlsbCBkZWZpbml0aW9uOiB7X1BJTExfREVGSU5JVElPTlNbcGlsbF9pZF19DQoNCkZvcm1hdCB5b3VyIHBpbGwgRVhBQ1RMWSBsaWtlIHRoaXM6DQoxLiBPbmUgc2VudGVuY2UgbmFtaW5nIHRoZSBjb25jZXB0IGluIHBsYWluIGxhbmd1YWdlIChubyBqYXJnb24pDQoyLiBPbmUgYW5hbG9neSBkcmF3biBmcm9tIHRoZWlyIHNwZWNpZmljIGpvYi9pbmR1c3RyeSAobm90IGdlbmVyaWMpDQozLiBPbmUgcXVlc3Rpb24gdGhhdCBjb25uZWN0cyB0aGlzIGNvbmNlcHQgdG8gc29tZXRoaW5nIHRoZXkgYWxyZWFkeSBkbyBhdCB3b3JrDQoNCkRvIE5PVCB1c2UgYnVsbGV0IHBvaW50cy4gV3JpdGUgaXQgYXMgbmF0dXJhbCBzcGVlY2guDQpSZXNwb25kIGluIHtsYW5ndWFnZX0uIiIiDQoNCiAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWw0KICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PXByb21wdCldKQ0KICAgIF0NCiAgICByZXR1cm4gX2NhbGwoY2xpZW50LCBjb250ZW50cykNCg0KDQpkZWYgcnVuX21hcCgNCiAgICBjbGllbnQ6IGdlbmFpLkNsaWVudCwNCiAgICBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwNCiAgICByb2xlOiBzdHIsDQogICAgc2VsZWN0ZWRfdXNlX2Nhc2U6IGRpY3QsDQogICAgcGlsbF9pZDogaW50LA0KICAgIGxhbmd1YWdlOiBzdHIsDQopIC0+IHN0cjoNCiAgICBwaWxsX2NvbmNlcHQgPSBfUElMTF9OQU1FUy5nZXQocGlsbF9pZCwgIlByb21wdGluZyIpDQogICAgcHJvbXB0ID0gZiIiIklucHV0OiB7cm9sZX0sIHtzZWxlY3RlZF91c2VfY2FzZX0sIHtwaWxsX2NvbmNlcHR9LCB7bGFuZ3VhZ2V9DQoNClRoZSB1c2VyIGlzIGEge3JvbGV9LiBUaGV5IGp1c3QgY29tcGxldGVkIHRoZWlyIGZpcnN0IEFJIHRhc2s6IHtzZWxlY3RlZF91c2VfY2FzZVsnbGFiZWwnXX0uDQpUaGV5IGxlYXJuZWQgYWJvdXQ6IHtwaWxsX2NvbmNlcHR9Lg0KDQpHZW5lcmF0ZSB0aGVpciBwZXJzb25hbCBBSSBtYXA6IGV4YWN0bHkgMyBuZXh0IHN0ZXBzIHRoZXkgY2FuIHRha2UgVEhJUyBXRUVLLg0KDQpSdWxlczoNCi0gRWFjaCBzdGVwIG11c3QgYmUgc3BlY2lmaWMgdG8gdGhlaXIgcm9sZS4gTm90IGdlbmVyaWMgYWR2aWNlLg0KLSBFYWNoIHN0ZXAgbXVzdCBiZSBzb21ldGhpbmcgdGhleSBjYW4gZG8gaW4gdW5kZXIgMzAgbWludXRlcy4NCi0gRWFjaCBzdGVwIG11c3QgYnVpbGQgb24gd2hhdCB0aGV5IGp1c3QgZGlkIOKAlCBub3Qgc3RhcnQgb3Zlci4NCi0gTm8gamFyZ29uLiBObyB0b29sIG5hbWVzIHRoZXkgZG9uJ3Qga25vdyB5ZXQuIE9uZSBmcmVlIHRvb2wgcmVjb21tZW5kYXRpb24gbWF4aW11bSBwZXIgc3RlcC4NCi0gRm9ybWF0IGFzIG51bWJlcmVkIGxpc3QuIE9uZSBzZW50ZW5jZSBwZXIgc3RlcC4gQWN0aW9uIHZlcmIgdG8gc3RhcnQuDQoNClJlc3BvbmQgaW4ge2xhbmd1YWdlfS4iIiINCg0KICAgIGNvbnRlbnRzID0gbGlzdChjb252ZXJzYXRpb25faGlzdG9yeSkgKyBbDQogICAgICAgIHR5cGVzLkNvbnRlbnQocm9sZT0idXNlciIsIHBhcnRzPVt0eXBlcy5QYXJ0KHRleHQ9cHJvbXB0KV0pDQogICAgXQ0KICAgIHJldHVybiBfY2FsbChjbGllbnQsIGNvbnRlbnRzKQ0K"
with open("chispa_core.py", "w", encoding="utf-8") as f:
    f.write(base64.b64decode(_core_b64).decode("utf-8"))

# server.py
_srv_b64 = "aW1wb3J0IG9zDQpmcm9tIGRvdGVudiBpbXBvcnQgbG9hZF9kb3RlbnYNCmZyb20gZmFzdGFwaSBpbXBvcnQgRmFzdEFQSSwgSFRUUEV4Y2VwdGlvbg0KZnJvbSBmYXN0YXBpLm1pZGRsZXdhcmUuY29ycyBpbXBvcnQgQ09SU01pZGRsZXdhcmUNCmZyb20gZmFzdGFwaS5yZXNwb25zZXMgaW1wb3J0IEZpbGVSZXNwb25zZQ0KZnJvbSBweWRhbnRpYyBpbXBvcnQgQmFzZU1vZGVsDQpmcm9tIHR5cGluZyBpbXBvcnQgQW55DQoNCmxvYWRfZG90ZW52KCkNCg0KZnJvbSBjaGlzcGFfY29yZSBpbXBvcnQgKA0KICAgIGJ1aWxkX2NsaWVudCwgYnVpbGRfaGlzdG9yeSwNCiAgICBydW5fZGlzY292ZXJ5LCBydW5fcGlja19jb25maXJtLCBydW5fd2luX29wZW4sDQogICAgcnVuX3dpbl9leGVjdXRlLCBydW5fd2luX2NvbmZpcm0sIHJ1bl9waWxsLCBydW5fbWFwLA0KICAgIHNlbGVjdF9waWxsLCBNT0RFTCwNCikNCg0KYXBwID0gRmFzdEFQSSh0aXRsZT0iQ2hpc3BhIEFQSSIpDQoNCmFwcC5hZGRfbWlkZGxld2FyZSgNCiAgICBDT1JTTWlkZGxld2FyZSwNCiAgICBhbGxvd19vcmlnaW5zPVsiKiJdLA0KICAgIGFsbG93X21ldGhvZHM9WyIqIl0sDQogICAgYWxsb3dfaGVhZGVycz1bIioiXSwNCikNCg0KX2NsaWVudCA9IGJ1aWxkX2NsaWVudChvcy5lbnZpcm9uLmdldCgiR09PR0xFX0FQSV9LRVkiLCAiIikpDQoNCg0KY2xhc3MgQ2hhdFJlcXVlc3QoQmFzZU1vZGVsKToNCiAgICBzdGFnZTogc3RyDQogICAgY29udmVyc2F0aW9uX2hpc3Rvcnk6IGxpc3RbZGljdF0NCiAgICB2YXJpYWJsZXM6IGRpY3Rbc3RyLCBBbnldID0ge30NCiAgICB1c2VyX21lc3NhZ2U6IHN0ciA9ICIiDQoNCg0KY2xhc3MgQ2hhdFJlc3BvbnNlKEJhc2VNb2RlbCk6DQogICAgcmVwbHk6IHN0cg0KICAgIHZhcmlhYmxlczogZGljdFtzdHIsIEFueV0NCiAgICBuZXh0X3N0YWdlOiBzdHINCiAgICBuZWVkc191c2VyX2lucHV0OiBib29sID0gVHJ1ZQ0KDQoNClZBTElEX1NUQUdFUyA9IHsNCiAgICAiZGlzY292ZXJ5IiwgInBpY2tfY29uZmlybSIsICJ3aW5fb3BlbiIsDQogICAgIndpbl9leGVjdXRlIiwgIndpbl9jb25maXJtIiwgInBpbGwiLCAibWFwIiwNCn0NCg0KDQpAYXBwLmdldCgiLyIpDQphc3luYyBkZWYgc2VydmVfZnJvbnRlbmQoKToNCiAgICBodG1sX3BhdGggPSBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKF9fZmlsZV9fKSwgImluZGV4Lmh0bWwiKQ0KICAgIHJldHVybiBGaWxlUmVzcG9uc2UoaHRtbF9wYXRoKQ0KDQoNCkBhcHAuZ2V0KCIvaGVhbHRoIikNCmRlZiBoZWFsdGgoKToNCiAgICByZXR1cm4geyJzdGF0dXMiOiAib2siLCAibW9kZWwiOiBNT0RFTH0NCg0KDQpAYXBwLnBvc3QoIi9hcGkvY2hhdCIsIHJlc3BvbnNlX21vZGVsPUNoYXRSZXNwb25zZSkNCmRlZiBjaGF0KHJlcTogQ2hhdFJlcXVlc3QpOg0KICAgIGlmIHJlcS5zdGFnZSBub3QgaW4gVkFMSURfU1RBR0VTOg0KICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTQyMiwgZGV0YWlsPWYiVW5rbm93biBzdGFnZToge3JlcS5zdGFnZX0uIFZhbGlkOiB7c29ydGVkKFZBTElEX1NUQUdFUyl9IikNCg0KICAgIGhpc3RvcnkgPSBidWlsZF9oaXN0b3J5KHJlcS5jb252ZXJzYXRpb25faGlzdG9yeSkNCiAgICB2ID0gZGljdChyZXEudmFyaWFibGVzKQ0KDQogICAgdHJ5Og0KICAgICAgICBpZiByZXEuc3RhZ2UgPT0gImRpc2NvdmVyeSI6DQogICAgICAgICAgICByZXN1bHQgPSBydW5fZGlzY292ZXJ5KF9jbGllbnQsIGhpc3RvcnkpDQogICAgICAgICAgICB2LnVwZGF0ZShyZXN1bHQpDQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PSIiLCB2YXJpYWJsZXM9diwgbmV4dF9zdGFnZT0icGlja19jb25maXJtIiwgbmVlZHNfdXNlcl9pbnB1dD1GYWxzZSkNCg0KICAgICAgICBpZiByZXEuc3RhZ2UgPT0gInBpY2tfY29uZmlybSI6DQogICAgICAgICAgICByZXBseSA9IHJ1bl9waWNrX2NvbmZpcm0oX2NsaWVudCwgaGlzdG9yeSwgdlsic2VsZWN0ZWRfdXNlX2Nhc2UiXSwgdlsicm9sZSJdLCB2WyJsYW5ndWFnZSJdKQ0KICAgICAgICAgICAgcmV0dXJuIENoYXRSZXNwb25zZShyZXBseT1yZXBseSwgdmFyaWFibGVzPXYsIG5leHRfc3RhZ2U9Indpbl9vcGVuIiwgbmVlZHNfdXNlcl9pbnB1dD1GYWxzZSkNCg0KICAgICAgICBpZiByZXEuc3RhZ2UgPT0gIndpbl9vcGVuIjoNCiAgICAgICAgICAgIHJlcGx5ID0gcnVuX3dpbl9vcGVuKF9jbGllbnQsIGhpc3RvcnksIHZbInNlbGVjdGVkX3VzZV9jYXNlIl0sIHZbInJvbGUiXSwgdlsibGFuZ3VhZ2UiXSkNCiAgICAgICAgICAgIHJldHVybiBDaGF0UmVzcG9uc2UocmVwbHk9cmVwbHksIHZhcmlhYmxlcz12LCBuZXh0X3N0YWdlPSJ3aW5fZXhlY3V0ZSIsIG5lZWRzX3VzZXJfaW5wdXQ9VHJ1ZSkNCg0KICAgICAgICBpZiByZXEuc3RhZ2UgPT0gIndpbl9leGVjdXRlIjoNCiAgICAgICAgICAgIHJlc3VsdCA9IHJ1bl93aW5fZXhlY3V0ZSgNCiAgICAgICAgICAgICAgICBfY2xpZW50LCBoaXN0b3J5LCB2WyJzZWxlY3RlZF91c2VfY2FzZSJdLA0KICAgICAgICAgICAgICAgIHYuZ2V0KCJ1c2VyX3Rhc2tfZGV0YWlscyIsIHJlcS51c2VyX21lc3NhZ2UpLA0KICAgICAgICAgICAgICAgIHZbInJvbGUiXSwgdlsibGFuZ3VhZ2UiXQ0KICAgICAgICAgICAgKQ0KICAgICAgICAgICAgdlsidGFza19vdXRwdXQiXSA9IHJlc3VsdFsib3V0cHV0Il0NCiAgICAgICAgICAgIHZbInRhc2tfb3V0cHV0X3N1bW1hcnkiXSA9IHJlc3VsdFsic3VtbWFyeSJdDQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PXJlc3VsdFsib3V0cHV0Il0sIHZhcmlhYmxlcz12LCBuZXh0X3N0YWdlPSJ3aW5fY29uZmlybSIsIG5lZWRzX3VzZXJfaW5wdXQ9VHJ1ZSkNCg0KICAgICAgICBpZiByZXEuc3RhZ2UgPT0gIndpbl9jb25maXJtIjoNCiAgICAgICAgICAgIHJlcGx5ID0gcnVuX3dpbl9jb25maXJtKF9jbGllbnQsIGhpc3RvcnksIHZbImxhbmd1YWdlIl0pDQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PXJlcGx5LCB2YXJpYWJsZXM9diwgbmV4dF9zdGFnZT0icGlsbCIsIG5lZWRzX3VzZXJfaW5wdXQ9RmFsc2UpDQoNCiAgICAgICAgaWYgcmVxLnN0YWdlID09ICJwaWxsIjoNCiAgICAgICAgICAgIHBpbGxfaWQgPSBzZWxlY3RfcGlsbCh2WyJzZWxlY3RlZF91c2VfY2FzZSJdKQ0KICAgICAgICAgICAgdlsicGlsbF9pZCJdID0gcGlsbF9pZA0KICAgICAgICAgICAgcmVwbHkgPSBydW5fcGlsbCgNCiAgICAgICAgICAgICAgICBfY2xpZW50LCBoaXN0b3J5LCBwaWxsX2lkLCB2WyJzZWxlY3RlZF91c2VfY2FzZSJdLA0KICAgICAgICAgICAgICAgIHZbInJvbGUiXSwgdlsibGFuZ3VhZ2UiXSwgdi5nZXQoInRhc2tfb3V0cHV0X3N1bW1hcnkiLCAiIikNCiAgICAgICAgICAgICkNCiAgICAgICAgICAgIHJldHVybiBDaGF0UmVzcG9uc2UocmVwbHk9cmVwbHksIHZhcmlhYmxlcz12LCBuZXh0X3N0YWdlPSJtYXAiLCBuZWVkc191c2VyX2lucHV0PVRydWUpDQoNCiAgICAgICAgaWYgcmVxLnN0YWdlID09ICJtYXAiOg0KICAgICAgICAgICAgcmVwbHkgPSBydW5fbWFwKF9jbGllbnQsIGhpc3RvcnksIHZbInJvbGUiXSwgdlsic2VsZWN0ZWRfdXNlX2Nhc2UiXSwgdi5nZXQoInBpbGxfaWQiLCAxKSwgdlsibGFuZ3VhZ2UiXSkNCiAgICAgICAgICAgIHJldHVybiBDaGF0UmVzcG9uc2UocmVwbHk9cmVwbHksIHZhcmlhYmxlcz12LCBuZXh0X3N0YWdlPSJkb25lIiwgbmVlZHNfdXNlcl9pbnB1dD1GYWxzZSkNCg0KICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToNCiAgICAgICAgcHJpbnQoZiJbQ2hpc3BhXSBBUEkgZXJyb3IgKHN0YWdlPXtyZXEuc3RhZ2V9KToge2V9IiwgZmx1c2g9VHJ1ZSkNCiAgICAgICAgcmV0dXJuIENoYXRSZXNwb25zZSgNCiAgICAgICAgICAgIHJlcGx5PSJHaXZlIG1lIGEgc2Vjb25kIOKAlCBJJ20gdGhpbmtpbmcuIiwNCiAgICAgICAgICAgIHZhcmlhYmxlcz12LA0KICAgICAgICAgICAgbmV4dF9zdGFnZT1yZXEuc3RhZ2UsDQogICAgICAgICAgICBuZWVkc191c2VyX2lucHV0PVRydWUsDQogICAgICAgICkNCg=="
with open("server.py", "w", encoding="utf-8") as f:
    f.write(base64.b64decode(_srv_b64).decode("utf-8"))

print("server files written: chispa_core.py, server.py")


In [ ]:
# Cell 9: Start FastAPI server
print("=== Cell 9: Start FastAPI server ===")
import subprocess, time

server_process = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)
print("FastAPI server started on port 8000.")


In [ ]:
# Cell 10: ngrok tunnel
print("=== Cell 10: Start ngrok tunnel ===")
!pip install pyngrok -q
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import os

ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000)
os.environ["CHISPA_PUBLIC_URL"] = public_url.public_url
print(f"Chispa is live at: {public_url.public_url}")


In [ ]:
# Cell 11: Inject ngrok URL into index.html
print("=== Cell 11: Inject ngrok URL into index.html ===")
import os, base64

if not os.path.exists("index.html"):
    print("index.html not found — regenerating...")
    _b64 = "..."
    html_content = base64.b64decode(_b64).decode("utf-8")
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html_content)
    print("index.html created.")

with open("index.html", "r", encoding="utf-8") as f:
    html = f.read()

ngrok_url = os.environ.get("CHISPA_PUBLIC_URL", "")
if not ngrok_url:
    print("WARNING: CHISPA_PUBLIC_URL not set — URL injection skipped")
else:
    html = html.replace(
        "window.CHISPA_API_URL = null",
        f"window.CHISPA_API_URL = '{ngrok_url}/api/chat'"
    )
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Injected API URL: {ngrok_url}/api/chat")
